# Avionics, operations and program evidence

This notebook exposes the reusable budget and logic models. The current master leaves many hardware selections and thresholds blank. Required inputs are therefore explicit, and demonstration fixtures are kept separate from vehicle evidence.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from hyperion.workspace import current_flight
from hyperion.verification import ROOT, activities, input_contracts

get_ipython().run_line_magic('matplotlib', 'inline')
plt.rcParams.update({'figure.figsize': (10, 4), 'axes.grid': True, 'grid.alpha': 0.2})
pd.set_option('display.max_colwidth', 100)

## Logic example: hold until all supplied conditions pass

This small example demonstrates boundary behavior, not an approved launch-commit rule. Use the actual thresholds, sensor age limits, voting and failure responses from the finalized requirement baseline before registering evidence.

In [2]:
from hyperion.systems import state_machine
logic_example = state_machine(initial_state='HOLD',
    events=[{'time_s':0, 'event':'ready', 'signals':{'all_ready':False}},
            {'time_s':1, 'event':'ready', 'signals':{'all_ready':True}},
            {'time_s':2, 'event':'anomaly', 'signals':{}},
            {'time_s':3, 'event':'ready', 'signals':{}}],
    transitions=[{'from':'HOLD','event':'ready','to':'READY','guards':{'all_ready':True}},
                 {'from':'READY','event':'anomaly','to':'HOLD','guards':{}},
                 {'from':'HOLD','event':'reset','to':'HOLD','guards':{}}],
    forbidden_states=['UNAUTHORIZED_RELEASE'])
display(pd.DataFrame(logic_example['history']))

,time_s,previous,state,event
0,0.0,HOLD,HOLD,ready
1,1.0,HOLD,READY,ready
2,2.0,READY,HOLD,anomaly
3,3.0,HOLD,HOLD,ready


## Available calculation inputs

Functions cover communication link/visibility, packet rate/latency, electrical energy and voltage drop, sensor covariance, control response/margins, finite-state sequences, fault trees, critical paths, cost, transport, lifting, environmental balances and weather screens.

`config/systems_examples.json` contains labelled unit-scale demonstrations. They are not loaded into the default verification run. Fill the activity-specific entry in `config/verification_inputs.json` using approved values and source references; add physical test records separately.

In [3]:
from hyperion import systems
contracts = input_contracts()
display(contracts[contracts.calculation.isin(systems.FUNCTIONS)])

,calculation,required_inputs,description
66,systems_rf_link,"tx_positions_m, rx_position_m, frequency_hz, tx_power_w, tx_gain_dbi, rx_gain_dbi, loss_db, nois...","Free-space RF link and ground antenna visibility; positions ECEF m, gains dBi, losses dB."
67,systems_satellite_visibility,"receiver_positions_m, satellite_positions_m, minimum_elevation_deg, earth_radius_m","GNSS geometry at simultaneous epochs; ECEF m arrays [T,3] and [T,S,3], mask degrees."
68,systems_telemetry,"streams, link_rate_bps, coding_rate, protocol_efficiency, propagation_delay_s, processing_delay_s","Packet budget; streams contain packet_bits and packets_per_s; times s, link rate bits/s."
69,systems_power_energy,"loads, capacity_wh, usable_fraction, distribution_efficiency, reserve_fraction, transfer_time_s,...","Electrical energy budget; loads contain power_w, duration_s, duty_fraction; capacity Wh, transfe..."
70,systems_dc_circuit,"source_voltage_v, source_resistance_ohm, conductor_resistance_ohm_m, one_way_length_m, load_curr...","Resistive two-wire DC voltage drop, heating, bolted fault and supplied clearing time; SI units."
71,systems_timing,"stages_s, deadline_s, jitter_s","Worst-case serial event/umbilical/command timing; stage durations, deadline and additive jitter ..."
72,systems_state_machine,"initial_state, events, transitions, forbidden_states","Evaluate explicit transition/guard logic; events {time_s,event,signals}, transitions {from,event..."
73,systems_launch_commit,"measurements, rules","Evaluate supplied launch/environment predicates; rules {name,measurement,operator,limit}; units ..."
74,systems_navigation,"transition_matrix, process_covariance, measurement_matrix, measurement_covariance, initial_covar...",Discrete linear Kalman covariance/bias propagation and final Gaussian Monte Carlo; use consisten...
75,systems_frequency_margins,"frequency_hz, open_loop_real, open_loop_imag","Sampled SISO open-loop gain/phase crossover margins; frequencies Hz, complex loop gain dimension..."


## Inspect activities for one subsystem

Change the domain or requirement filter to locate the required calculation, test or inspection. The complete register includes all 373 tiered requirements. Document reviews, physical demonstrations and hardware tests use the evidence importer, which validates identifiers and file integrity without fabricating results.

In [4]:
register = activities()
display(register[register.requirement_id.str.startswith('AVI.3.COM')][
    ['activity_id','activity','kernel','open_parameters']])

,activity_id,activity,kernel,open_parameters
411,AVI.3.COM.01.A01,RF link budget and antenna-visibility sweep.,systems_rf_link,TBD-AVI-01
412,AVI.3.COM.01.A02,Radio-frequency and link-performance tests.,NaN,TBD-AVI-01
413,AVI.3.COM.02.A01,Telemetry packet-rate and bandwidth budget.,systems_telemetry,
414,AVI.3.COM.02.A02,Telemetry continuity functional tests.,NaN,
415,AVI.3.COM.03.A01,Engine-command telemetry packet-rate budget.,systems_telemetry,
416,AVI.3.COM.03.A02,Command-to-telemetry end-to-end tests.,NaN,
417,AVI.3.COM.04.A01,Link-recovery state-transition and timing model.,systems_state_logic,"TBD-AVI-23, TBD-AVI-24"
418,AVI.3.COM.04.A02,Communication-outage fault-injection tests.,NaN,"TBD-AVI-23, TBD-AVI-24"
419,AVI.3.COM.05.A01,GNSS satellite-visibility geometry sweep.,systems_satellite_visibility,TBD-AVI-27
420,AVI.3.COM.05.A02,GNSS receiver update-rate performance tests.,NaN,TBD-AVI-27
